In [0]:
# initial load for volumn to source table 
from pyspark.sql.functions import current_timestamp, input_file_name, col

volume_path = "/Volumes/dbt_airbnb/source/source_data/"
catalog_name = "dbt_airbnb"
source_schema = "source"
full_schema_path = f"{catalog_name}.{source_schema}"

# 1. Schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {full_schema_path}")

# 2.Volume data
files = dbutils.fs.ls(volume_path)

for file in files:
    if file.name.endswith(".csv"):
        table_name = file.name.replace(".csv", "")
        
        # when read csv, we need to specify schema
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "false") \
            .load(file.path)
        
        df_final = df.withColumn("ingested_at", current_timestamp()) \
                     .withColumn("source_file", col("_metadata.file_path"))
        
        # write to table
        df_final.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"{full_schema_path}.{table_name}")

print(f"✅ table {table_name} has been created in Unity Catalog metadata。")

✅ 表 listings 已成功入库并包含最新的 Unity Catalog 元数据。


In [0]:
# purpose for incre_load
from pyspark.sql.functions import current_timestamp, col

# set volume path
volume_path = "/Volumes/dbt_airbnb/source/source_data/incre_load/"
catalog_name = "dbt_airbnb"
source_schema = "source"
full_schema_path = f"{catalog_name}.{source_schema}"

files = dbutils.fs.ls(volume_path)

for file in files:
    if file.name.endswith(".csv"):
        
        # union write to table listings ( listing_incre.csv）
        table_name = "listings"
        
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "false") \
            .load(file.path)
        
        df_final = df.withColumn("ingested_at", current_timestamp()) \
                     .withColumn("source_file", col("_metadata.file_path"))
        
        # 👉 append
        df_final.write.format("delta") \
            .mode("append") \
            .saveAsTable(f"{full_schema_path}.{table_name}")

print("✅ data has been appended to table listings")

✅ 增量数据已追加


In [0]:
%sql
select * from dbt_airbnb.source.listings
order by 1

listing_id,host_id,property_type,room_type,city,country,accommodates,bedrooms,bathrooms,price_per_night,created_at,ingested_at,source_file
1,150,House,Entire home,Melbourne,Australia,3,1,2,84,2025-12-26 14:15:54.011160,2026-06-14T12:32:56.116Z,dbfs:/Volumes/dbt_airbnb/source/source_data/incre_load/listing_incre.csv
1,150,House,Entire home,West Lisafort,Malta,3,1,2,84,2025-12-26 14:15:54.011160,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
10,131,Apartment,Entire home,Carlsonside,Saudi Arabia,4,3,1,231,2025-12-26 14:15:54.011160,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
100,177,Apartment,Private room,Hansenport,Djibouti,1,3,2,63,2025-12-26 14:15:54.011160,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
101,173,House,Private room,Elliottborough,Sao Tome and Principe,5,2,3,91,2025-12-26 14:15:54.011160,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
102,8,House,Private room,Troyville,Uganda,1,2,3,84,2025-12-26 14:15:54.011160,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
103,67,Condo,Entire home,Leemouth,Uganda,3,2,1,216,2025-12-26 14:15:54.011160,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
104,194,House,Private room,North Stephanie,Sao Tome and Principe,4,4,3,73,2025-12-26 14:15:54.011160,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
105,183,Apartment,Entire home,Owensshire,French Guiana,5,2,1,246,2025-12-26 14:15:54.011160,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
106,13,Apartment,Private room,Port Jenniferhaven,Lesotho,6,2,2,256,2025-12-26 14:15:54.011160,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv


In [0]:
%sql
select count(*) from dbt_airbnb.source.listings
limit 5

count(*)
500


In [0]:
%sql
select * from dbt_airbnb.source.listings
where listing_id in (1, 501)

listing_id,host_id,property_type,room_type,city,country,accommodates,bedrooms,bathrooms,price_per_night,created_at,ingested_at,source_file
1,150,House,Entire home,West Lisafort,Malta,3,1,2,84,2025-12-26 14:15:54.011160,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
1,150,House,Entire home,Melbourne,Australia,3,1,2,84,2025-12-26 14:15:54.011160,2026-06-14T12:32:56.116Z,dbfs:/Volumes/dbt_airbnb/source/source_data/incre_load/listing_incre.csv
501,91,Apartment,Private room,Sydney,Australia,1,2,3,140,2025-12-26 14:15:54.011160,2026-06-14T12:32:56.116Z,dbfs:/Volumes/dbt_airbnb/source/source_data/incre_load/listing_incre.csv


In [0]:
%sql
select * from dbt_airbnb.bronze.stg_bookings

booking_id,listing_id,booking_date,nights_booked,booking_amount,cleaning_fee,service_fee,booking_status,created_at,ingested_at,source_file
ef0bd262-f88d-477f-aeba-9c69cc72c051,379,2024-12-25,3,591,70,23,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
490685e5-1b70-429b-84ac-d605509467af,111,2025-02-18,3,171,45,19,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
35278efe-9927-4dfd-ae16-693d5687403b,317,2025-03-24,6,780,73,42,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
e821bada-b726-494a-b423-d179d6ca5ef7,253,2025-06-24,13,1131,38,28,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
67af9fda-6cce-4f90-b938-54990993a668,113,2025-04-29,11,1947,53,31,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
07a51ddc-85e0-4faa-aa74-3d90f49940f8,375,2025-08-28,6,1380,46,23,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
8ed5abfe-e564-4ec4-b7f3-bceed676e0a3,230,2025-08-02,1,192,29,48,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
cb9f00e9-0ac1-4d6b-9467-92e04869dbdc,372,2025-02-03,6,1518,50,41,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
40396917-9111-4de6-87f8-56d562f98666,425,2025-12-20,5,800,50,26,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
816eb804-addb-4156-bfc3-dc5b7f6565f0,277,2025-08-03,5,415,24,12,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv


In [0]:
%sql
select * from dbt_airbnb.bronze.stg_listings
limit 5

listing_id,host_id,property_type,room_type,city,country,accommodates,bedrooms,bathrooms,price_per_night,created_at,ingested_at,source_file
1,150,House,Entire home,West Lisafort,Malta,3,1,2,84,2025-12-26 14:15:54.011160,2026-04-20T03:24:21.296Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
2,91,Apartment,Private room,North Cherylberg,Western Sahara,1,2,3,140,2025-12-26 14:15:54.011160,2026-04-20T03:24:21.296Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
3,65,Condo,Entire home,Whitneyfort,Martinique,1,4,3,215,2025-12-26 14:15:54.011160,2026-04-20T03:24:21.296Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
4,48,Condo,Entire home,New Crystal,Argentina,1,2,1,165,2025-12-26 14:15:54.011160,2026-04-20T03:24:21.296Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
5,63,Apartment,Private room,Churchmouth,Maldives,5,2,3,136,2025-12-26 14:15:54.011160,2026-04-20T03:24:21.296Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv


In [0]:
%sql
SELECT 'stg_bookings' AS table_name, COUNT(*) AS count FROM dbt_airbnb.bronze.stg_bookings
UNION ALL
SELECT 'stg_hosts' AS table_name, COUNT(*) AS count FROM dbt_airbnb.bronze.stg_hosts
UNION ALL
SELECT 'stg_listings' AS table_name, COUNT(*) AS count FROM dbt_airbnb.bronze.stg_listings;

table_name,count
stg_bookings,5000
stg_hosts,200
stg_listings,500


In [0]:
%sql
select * from dbt_airbnb.bronze.stg_hosts
limit 3

host_id,host_name,host_since,is_superhost,response_rate,created_at,ingested_at,source_file
1,Timothy Parker,2018-03-20,False,99,2025-12-26 14:15:54.011160,2026-04-20T03:24:17.822Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv
2,Hannah Evans,2024-01-01,False,95,2025-12-26 14:15:54.011160,2026-04-20T03:24:17.822Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv
3,Crystal Green,2016-08-06,False,74,2025-12-26 14:15:54.011160,2026-04-20T03:24:17.822Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv


In [0]:
%sql
select * from dbt_airbnb.bronze.stg_listings
where listing_id in (1, 501)
order by 1

listing_id,host_id,property_type,room_type,city,country,accommodates,bathrooms,bedrooms,price_per_night,created_at,ingested_at,source_file
1,150,House,Entire home,Melbourne,Australia,3,2.0,1.0,84.0,2025-12-26T14:15:54.011Z,2026-06-14T12:32:56.116Z,dbfs:/Volumes/dbt_airbnb/source/source_data/incre_load/listing_incre.csv
501,91,Apartment,Private room,Sydney,Australia,1,3.0,2.0,140.0,2025-12-26T14:15:54.011Z,2026-06-14T12:32:56.116Z,dbfs:/Volumes/dbt_airbnb/source/source_data/incre_load/listing_incre.csv


In [0]:
%sql
select * from dbt_airbnb.silver.silver_hosts
limit 5;

host_id,host_name,host_since,is_superhost,response_rate,response_rate_tag,created_at,ingested_at,source_file
1,Timothy_Parker,2018-03-20,false,99.0,very_good,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv
2,Hannah_Evans,2024-01-01,false,95.0,good,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv
3,Crystal_Green,2016-08-06,false,74.0,fair,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv
4,Kevin_Johnson,2020-02-25,false,100.0,very_good,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv
5,Monica_Johnson,2024-11-11,false,77.0,fair,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv


In [0]:
%sql
select * from dbt_airbnb.silver.silver_bookings
limit 5;

booking_id,listing_id,booking_date,nights_booked,booking_amount,cleaning_fee,service_fee,booking_amount_rounded,total_booking_amount,booking_status,created_at,ingested_at,source_file
ef0bd262-f88d-477f-aeba-9c69cc72c051,379,2024-12-25,3,591,70,23,1773.0,1866.0,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
490685e5-1b70-429b-84ac-d605509467af,111,2025-02-18,3,171,45,19,513.0,577.0,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
35278efe-9927-4dfd-ae16-693d5687403b,317,2025-03-24,6,780,73,42,4680.0,4795.0,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
e821bada-b726-494a-b423-d179d6ca5ef7,253,2025-06-24,13,1131,38,28,14703.0,14769.0,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv
67af9fda-6cce-4f90-b938-54990993a668,113,2025-04-29,11,1947,53,31,21417.0,21501.0,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv


In [0]:
%sql
SELECT 
    booking_id, 
    COUNT(*) as occurrences
FROM dbt_airbnb.silver.silver_bookings
GROUP BY 1
HAVING COUNT(*) > 1;

booking_id,occurrences


In [0]:
%sql
select * from dbt_airbnb.silver.silver_listings
limit 5;

listing_id,host_id,property_type,room_type,city,country,accommodates,bathrooms,bedrooms,price_per_night,price_per_night_tag,created_at,ingested_at,source_file
1,150,House,Entire home,West Lisafort,Malta,3,2,1,84,low,2025-12-26 14:15:54.011160,2026-04-20T03:24:21.296Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
2,91,Apartment,Private room,North Cherylberg,Western Sahara,1,3,2,140,medium,2025-12-26 14:15:54.011160,2026-04-20T03:24:21.296Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
3,65,Condo,Entire home,Whitneyfort,Martinique,1,3,4,215,high,2025-12-26 14:15:54.011160,2026-04-20T03:24:21.296Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
4,48,Condo,Entire home,New Crystal,Argentina,1,1,2,165,medium,2025-12-26 14:15:54.011160,2026-04-20T03:24:21.296Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv
5,63,Apartment,Private room,Churchmouth,Maldives,5,3,2,136,medium,2025-12-26 14:15:54.011160,2026-04-20T03:24:21.296Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv


In [0]:
%sql
select * from dbt_airbnb.gold.obt
where listing_id in (1, 501)
order by 1

booking_id,listing_id,booking_date,nights_booked,booking_amount,cleaning_fee,service_fee,booking_amount_rounded,total_booking_amount,booking_status,booking_created_at,booking_ingested_at,booking_source_file,property_type,room_type,city,country,accommodates,bathrooms,bedrooms,price_per_night,price_per_night_tag,listing_created_at,listing_ingested_at,host_id,host_name,host_since,is_superhost,response_rate,response_rate_tag,host_created_at,host_ingested_at
7c0cf414-0d2e-4696-b14b-3dbc6bee84f7,1,2025-02-12,5,430,36,46,2150.0,2232.0,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Entire home,Melbourne,Australia,3,2,1,84,low,2025-12-26 14:15:54.011160,2026-04-21T03:06:18.555Z,150,Lori_Mckinney,2022-10-03,True,79,fair,2025-12-26 14:15:54.011160,2026-04-20T03:24:17.822Z
8a94e10d-5352-4c2c-9a0c-5467b8fde720,1,2025-09-01,7,924,55,43,6468.0,6566.0,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Entire home,Melbourne,Australia,3,2,1,84,low,2025-12-26 14:15:54.011160,2026-04-21T03:06:18.555Z,150,Lori_Mckinney,2022-10-03,True,79,fair,2025-12-26 14:15:54.011160,2026-04-20T03:24:17.822Z
8e68b9a3-fc88-42aa-b93e-86bcf7483dc2,1,2025-11-21,8,848,56,41,6784.0,6881.0,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Entire home,Melbourne,Australia,3,2,1,84,low,2025-12-26 14:15:54.011160,2026-04-21T03:06:18.555Z,150,Lori_Mckinney,2022-10-03,True,79,fair,2025-12-26 14:15:54.011160,2026-04-20T03:24:17.822Z
a1607f1d-de5f-4fa2-9b5d-ffe80266092e,1,2025-08-18,10,1100,54,47,11000.0,11101.0,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Entire home,Melbourne,Australia,3,2,1,84,low,2025-12-26 14:15:54.011160,2026-04-21T03:06:18.555Z,150,Lori_Mckinney,2022-10-03,True,79,fair,2025-12-26 14:15:54.011160,2026-04-20T03:24:17.822Z
a969aac7-0cb0-4cad-95e3-f4be78e01b2b,1,2025-10-13,13,1274,80,21,16562.0,16663.0,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Entire home,Melbourne,Australia,3,2,1,84,low,2025-12-26 14:15:54.011160,2026-04-21T03:06:18.555Z,150,Lori_Mckinney,2022-10-03,True,79,fair,2025-12-26 14:15:54.011160,2026-04-20T03:24:17.822Z
b02fbd1a-1b6a-4161-b433-a56cd70b3553,1,2025-03-26,6,906,72,18,5436.0,5526.0,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Entire home,Melbourne,Australia,3,2,1,84,low,2025-12-26 14:15:54.011160,2026-04-21T03:06:18.555Z,150,Lori_Mckinney,2022-10-03,True,79,fair,2025-12-26 14:15:54.011160,2026-04-20T03:24:17.822Z
bdc051a5-a5c9-473d-9c41-7181ce046590,1,2025-11-08,13,1482,44,20,19266.0,19330.0,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Entire home,Melbourne,Australia,3,2,1,84,low,2025-12-26 14:15:54.011160,2026-04-21T03:06:18.555Z,150,Lori_Mckinney,2022-10-03,True,79,fair,2025-12-26 14:15:54.011160,2026-04-20T03:24:17.822Z
ce6b65fe-3214-44b5-b03a-3c9759ee2a0f,1,2025-08-24,4,220,38,44,880.0,962.0,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Entire home,Melbourne,Australia,3,2,1,84,low,2025-12-26 14:15:54.011160,2026-04-21T03:06:18.555Z,150,Lori_Mckinney,2022-10-03,True,79,fair,2025-12-26 14:15:54.011160,2026-04-20T03:24:17.822Z
da546ad1-5b8d-4d0b-830e-058f6edf8aa9,1,2025-07-16,9,1035,31,32,9315.0,9378.0,cancelled,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Entire home,Melbourne,Australia,3,2,1,84,low,2025-12-26 14:15:54.011160,2026-04-21T03:06:18.555Z,150,Lori_Mckinney,2022-10-03,True,79,fair,2025-12-26 14:15:54.011160,2026-04-20T03:24:17.822Z
fc0

In [0]:
%sql
SELECT booking_id, COUNT(*)
FROM dbt_airbnb.gold.obt
GROUP BY booking_id
HAVING COUNT(*) > 1
LIMIT 10;

booking_id,COUNT(*)


In [0]:
%sql
select * from dbt_airbnb.gold.obt
limit 5;

booking_id,listing_id,booking_date,nights_booked,booking_amount,cleaning_fee,service_fee,total_amount,booking_status,booking_created_at,booking_ingested_at,booking_source_file,property_type,room_type,city,country,accommodates,bathrooms,bedrooms,price_per_night,price_per_night_tag,listing_created_at,listing_ingested_at,host_id,host_name,host_since,is_superhost,response_rate,response_rate_tag,host_created_at,host_ingested_at
ef0bd262-f88d-477f-aeba-9c69cc72c051,379,2024-12-25,3,591.0,70.0,23.0,1773.0,confirmed,2025-12-26T14:15:54.011Z,2026-06-09T23:26:08.683Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,Condo,Entire home,South Randybury,Cameroon,3,2.0,3.0,192.0,medium,2025-12-26T14:15:54.011Z,2026-06-09T23:26:20.518Z,78,Tina_Henry,2023-07-07,true,82.0,good,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z
490685e5-1b70-429b-84ac-d605509467af,111,2025-02-18,3,171.0,45.0,19.0,513.0,confirmed,2025-12-26T14:15:54.011Z,2026-06-09T23:26:08.683Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Entire home,West Alexandriaside,Lebanon,6,2.0,4.0,204.0,high,2025-12-26T14:15:54.011Z,2026-06-09T23:26:20.518Z,15,Kenneth_Roach,2016-06-28,true,93.0,good,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z
35278efe-9927-4dfd-ae16-693d5687403b,317,2025-03-24,6,780.0,73.0,42.0,4680.0,confirmed,2025-12-26T14:15:54.011Z,2026-06-09T23:26:08.683Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Entire home,Josephfort,Slovenia,3,1.0,1.0,173.0,medium,2025-12-26T14:15:54.011Z,2026-06-09T23:26:20.518Z,7,Gerald_Hunt,2022-01-21,true,99.0,very_good,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z
e821bada-b726-494a-b423-d179d6ca5ef7,253,2025-06-24,13,1131.0,38.0,28.0,14703.0,cancelled,2025-12-26T14:15:54.011Z,2026-06-09T23:26:08.683Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Private room,Cameronberg,Korea,1,1.0,4.0,241.0,high,2025-12-26T14:15:54.011Z,2026-06-09T23:26:20.518Z,42,David_Shepard,2017-08-31,false,100.0,very_good,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z
67af9fda-6cce-4f90-b938-54990993a668,113,2025-04-29,11,1947.0,53.0,31.0,21417.0,cancelled,2025-12-26T14:15:54.011Z,2026-06-09T23:26:08.683Z,dbfs:/Volumes/dbt_airbnb/source/source_data/bookings.csv,House,Private room,South Crystal,Northern Mariana Islands,2,3.0,1.0,246.0,high,2025-12-26T14:15:54.011Z,2026-06-09T23:26:20.518Z,108,Jacob_Campbell,2016-05-07,false,88.0,good,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z


In [0]:
%sql
select * from dbt_airbnb.gold.gold_fact_bookings
limit 3


booking_id,listing_id,booking_date,nights_booked,booking_amount,cleaning_fee,service_fee,booking_amount_rounded,total_booking_amount,booking_status,booking_created_at,booking_last_sync_time
ef0bd262-f88d-477f-aeba-9c69cc72c051,379,2024-12-25,3,591,70,23,1773.0,1866.0,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z
490685e5-1b70-429b-84ac-d605509467af,111,2025-02-18,3,171,45,19,513.0,577.0,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z
35278efe-9927-4dfd-ae16-693d5687403b,317,2025-03-24,6,780,73,42,4680.0,4795.0,confirmed,2025-12-26 14:15:54.011160,2026-04-20T03:24:13.832Z


In [0]:
%sql
select * from dbt_airbnb.gold.gold_fact_bookings
where listing_id in (1, 501)
order by 1

booking_id,listing_id,city,property_type,booking_date,total_booking_amount,booking_status,last_sync_time
7c0cf414-0d2e-4696-b14b-3dbc6bee84f7,1,Melbourne,House,2025-02-12,2232.0,cancelled,2026-04-20T03:24:13.832Z
8a94e10d-5352-4c2c-9a0c-5467b8fde720,1,Melbourne,House,2025-09-01,6566.0,confirmed,2026-04-20T03:24:13.832Z
8e68b9a3-fc88-42aa-b93e-86bcf7483dc2,1,Melbourne,House,2025-11-21,6881.0,confirmed,2026-04-20T03:24:13.832Z
a1607f1d-de5f-4fa2-9b5d-ffe80266092e,1,Melbourne,House,2025-08-18,11101.0,cancelled,2026-04-20T03:24:13.832Z
a969aac7-0cb0-4cad-95e3-f4be78e01b2b,1,Melbourne,House,2025-10-13,16663.0,confirmed,2026-04-20T03:24:13.832Z
b02fbd1a-1b6a-4161-b433-a56cd70b3553,1,Melbourne,House,2025-03-26,5526.0,confirmed,2026-04-20T03:24:13.832Z
bdc051a5-a5c9-473d-9c41-7181ce046590,1,Melbourne,House,2025-11-08,19330.0,cancelled,2026-04-20T03:24:13.832Z
ce6b65fe-3214-44b5-b03a-3c9759ee2a0f,1,Melbourne,House,2025-08-24,962.0,cancelled,2026-04-20T03:24:13.832Z
da546ad1-5b8d-4d0b-830e-058f6edf8aa9,1,Melbourne,House,2025-07-16,9378.0,cancelled,2026-04-20T03:24:13.832Z
fc0d86df-c62d-4bae-ae9f-2d4c7bcbe66a,1,Melbourne,House,2025-11-02,33701.0,confirmed,2026-04-20T03:24:13.832Z


In [0]:
%sql
select count(*) 
from dbt_airbnb.snapshots.hosts_snapshot

count(*)
200


In [0]:
%sql
select * from dbt_airbnb.snapshots.hosts_snapshot
limit 5

host_id,host_name,host_since,is_superhost,response_rate,created_at,ingested_at,source_file,dbt_scd_id,dbt_updated_at,dbt_valid_from,dbt_valid_to
1,Timothy Parker,2018-03-20,false,99.0,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv,ea531b0bb95dddc3ef153cff01da39f3,2026-06-09T23:26:16.006Z,2026-06-09T23:26:16.006Z,null
2,Hannah Evans,2024-01-01,false,95.0,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv,1a30203427908fde540e867d98561ce8,2026-06-09T23:26:16.006Z,2026-06-09T23:26:16.006Z,null
3,Crystal Green,2016-08-06,false,74.0,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv,5e251f2a2254b4ae01996cb7a0166966,2026-06-09T23:26:16.006Z,2026-06-09T23:26:16.006Z,null
4,Kevin Johnson,2020-02-25,false,100.0,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv,553bbb70c2039a282cf5de3c374a8734,2026-06-09T23:26:16.006Z,2026-06-09T23:26:16.006Z,null
5,Monica Johnson,2024-11-11,false,77.0,2025-12-26T14:15:54.011Z,2026-06-09T23:26:16.006Z,dbfs:/Volumes/dbt_airbnb/source/source_data/hosts.csv,d69c4cbda17c61f2e6e4feb1c23af15c,2026-06-09T23:26:16.006Z,2026-06-09T23:26:16.006Z,null


In [0]:
%sql
select * from dbt_airbnb.snapshots.listings_snapshot
where listing_id in (1, 501)
order by 1

listing_id,host_id,property_type,room_type,city,country,accommodates,bathrooms,bedrooms,price_per_night,created_at,ingested_at,source_file,dbt_scd_id,dbt_updated_at,dbt_valid_from,dbt_valid_to
1,150,House,Entire home,Melbourne,Australia,3,2.0,1.0,84.0,2025-12-26T14:15:54.011Z,2026-06-14T12:32:56.116Z,dbfs:/Volumes/dbt_airbnb/source/source_data/incre_load/listing_incre.csv,fc7998dc8aa06f77d5e6c2d2062ce577,2026-06-14T12:32:56.116Z,2026-06-14T12:32:56.116Z,null
1,150,House,Entire home,West Lisafort,Malta,3,2.0,1.0,84.0,2025-12-26T14:15:54.011Z,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv,5a95d141af167eaf67e3e7425354d294,2026-06-09T23:26:20.518Z,2026-06-09T23:26:20.518Z,2026-06-14T12:32:56.116Z
501,91,Apartment,Private room,Sydney,Australia,1,3.0,2.0,140.0,2025-12-26T14:15:54.011Z,2026-06-14T12:32:56.116Z,dbfs:/Volumes/dbt_airbnb/source/source_data/incre_load/listing_incre.csv,cdd982f7bf585d9e01e778a19fb16718,2026-06-14T12:32:56.116Z,2026-06-14T12:32:56.116Z,null


In [0]:
%sql
select count(*) 
from dbt_airbnb.snapshots.listings_snapshot

count(*)
500


In [0]:
%sql
select * from dbt_airbnb.snapshots.listings_snapshot
limit 5

listing_id,host_id,property_type,room_type,city,country,accommodates,bathrooms,bedrooms,price_per_night,created_at,ingested_at,source_file,dbt_scd_id,dbt_updated_at,dbt_valid_from,dbt_valid_to
1,150,House,Entire home,West Lisafort,Malta,3,2.0,1.0,84.0,2025-12-26T14:15:54.011Z,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv,5a95d141af167eaf67e3e7425354d294,2026-06-09T23:26:20.518Z,2026-06-09T23:26:20.518Z,null
2,91,Apartment,Private room,North Cherylberg,Western Sahara,1,3.0,2.0,140.0,2025-12-26T14:15:54.011Z,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv,f369fb0407db5081f323b586208aa641,2026-06-09T23:26:20.518Z,2026-06-09T23:26:20.518Z,null
3,65,Condo,Entire home,Whitneyfort,Martinique,1,3.0,4.0,215.0,2025-12-26T14:15:54.011Z,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv,aad87908cb013d75e5d058ca7eea5abd,2026-06-09T23:26:20.518Z,2026-06-09T23:26:20.518Z,null
4,48,Condo,Entire home,New Crystal,Argentina,1,1.0,2.0,165.0,2025-12-26T14:15:54.011Z,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv,da3f99c9bfddb674004df72406a15f9b,2026-06-09T23:26:20.518Z,2026-06-09T23:26:20.518Z,null
5,63,Apartment,Private room,Churchmouth,Maldives,5,3.0,2.0,136.0,2025-12-26T14:15:54.011Z,2026-06-09T23:26:20.518Z,dbfs:/Volumes/dbt_airbnb/source/source_data/listings.csv,4f9238eacf2aa07471d107b6f7ebda5e,2026-06-09T23:26:20.518Z,2026-06-09T23:26:20.518Z,null
